# Merging normative models

In many real-world scenarios, normative models are trained independently at different sites or institutions. **Merging** allows us to combine these separately trained models into a single global model — without ever sharing the raw data.

`NormativeModel.merge()` works by:
1. **Synthesizing** data from each fitted model (using `synthesize()` internally)
2. **Pooling** the synthetic datasets together
3. **Refitting** a single model on the combined synthetic data

The result is a unified normative model that covers all the batch effects (e.g., sites, scanners) from every contributing model.

For a more in-depth, step-by-step tutorial on federated normative modeling with privacy guarantees, see **Tutorial 12 — Federated Normative Modeling**.

In [1]:
import copy
import logging
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns

import pcntoolkit.util.output
from pcntoolkit import (
    HBR,
    BsplineBasisFunction,
    NormalLikelihood,
    NormativeModel,
    NormData,
    load_fcon1000,
    make_prior,
)

sns.set_style("darkgrid")

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")

pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
pcntoolkit.util.output.Output.set_show_messages(False)

## Load and split data

We load the FCON-1000 dataset and partition its ~21 sites into three random groups of roughly equal size. Each group simulates an independently collected dataset at a different institution.

In [2]:
# Download an example dataset
norm_data: NormData = load_fcon1000()

# Select only a few features
features_to_model = [
    "WM-hypointensities",
    "Right-Lateral-Ventricle",
    # "Right-Amygdala",
    # "CortexVol",
]
norm_data = norm_data.sel({"response_vars": features_to_model})

all_sites = np.unique(norm_data.batch_effects.sel(batch_effect_dims="site").values)

# split all_sites into three random groups of 7 sites
np.random.shuffle(all_sites)
group1 = all_sites[:7]
group2 = all_sites[7:14]
group3 = all_sites[14:]
print(f"Group 1: {group1}")
print(f"Group 2: {group2}")
print(f"Group 3: {group3}")

data_group1, data_group23 = norm_data.batch_effects_split({"site": group1}, names=("group1", "group23"))
data_group2, data_group3 = data_group23.batch_effects_split({"site": group2}, names=("group2", "group3"))

Group 1: ['Leiden_2180' 'Atlanta' 'Berlin_Margulies' 'NewYork_a' 'Munchen'
 'AnnArbor_a' 'Leiden_2200']
Group 2: ['Queensland' 'SaintLouis' 'Milwaukee_b' 'Newark' 'PaloAlto'
 'Beijing_Zang' 'Bangor']
Group 3: ['AnnArbor_b' 'ICBM' 'Pittsburgh' 'Cambridge_Buckner' 'Oxford' 'Cleveland'
 'Oulu' 'NewYork_a_ADHD' 'Baltimore']


## Configure the model template

We define a shared HBR model configuration that every site will use. This ensures that all models share the same prior structure and hyperparameters, so the merge is well-defined.

In [3]:
mu = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 10.0)),
    intercept=make_prior(
        random=True,
        mu=make_prior(dist_name="Normal", dist_params=(0.0, 1.0)),
        sigma=make_prior(dist_name="Normal", dist_params=(0.0, 1.0), mapping="softplus", mapping_params=(0.0, 3.0)),
    ),
    basis_function=BsplineBasisFunction(basis_column=0, nknots=5, degree=3),
)
sigma = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 2.0)),
    intercept=make_prior(dist_name="Normal", dist_params=(1.0, 1.0)),
    basis_function=BsplineBasisFunction(basis_column=0, nknots=5, degree=3),
    mapping="softplus",
    mapping_params=(0.0, 3.0),
)

likelihood = NormalLikelihood(mu, sigma)

template_hbr = HBR(
    name="template",
    cores=16,
    progressbar=True,
    draws=1500,
    tune=500,
    chains=4,
    nuts_sampler="nutpie",
    likelihood=likelihood,
)

model = NormativeModel(
    template_regression_model=template_hbr,
    savemodel=True,
    evaluate_model=True,
    saveresults=True,
    saveplots=True,
    save_dir="resources/hbr_normal/save_dir",
    inscaler="standardize",
    outscaler="standardize",
)

## Train separate models

Each group trains its own model independently, as if the data lived at different institutions. No data is shared between groups — only the resulting model files will be exchanged later.

In [4]:
model1 = copy.deepcopy(model)
model1.save_dir = "resources/hbr_merge/model1"
model2 = copy.deepcopy(model)
model2.save_dir = "resources/hbr_merge/model2"
model3 = copy.deepcopy(model)
model3.save_dir = "resources/hbr_merge/model3"

model1.fit(data_group1)
model2.fit(data_group2)
model3.fit(data_group3)

c:\Users\kontsi\AppData\Local\anaconda3\envs\.ptk-dev\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.15,127
,2000,0,0.14,127
,2000,0,0.15,127
,2000,1,0.14,31


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.16,63
,2000,0,0.15,31
,2000,0,0.15,31
,2000,0,0.17,31


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,3,0.12,63
,2000,1,0.13,63
,2000,1,0.13,63
,2000,9,0.12,63


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.16,63
,2000,0,0.14,31
,2000,1,0.14,31
,2000,0,0.14,31


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.11,255
,2000,0,0.09,63
,2000,0,0.11,63
,2000,0,0.09,63


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.14,127
,2000,0,0.14,31
,2000,0,0.14,63
,2000,0,0.13,63


In [5]:
# model1 = NormativeModel.load(path="resources/hbr_merge/model1")
# model2 = NormativeModel.load(path="resources/hbr_merge/model2")
# model3 = NormativeModel.load(path="resources/hbr_merge/model3")

## Merge the models

`NormativeModel.merge()` accepts a list of fitted models (or paths to saved models on disk). Internally it synthesizes data from each model, concatenates the synthetic datasets, and refits a single global model.

The merged model can now predict on observations from **any** of the original sites.

In [6]:
# We can pass a list of models or paths to the merge function.
merged_model = NormativeModel.merge(
    save_dir="resources/hbr_merge/merged_model", models=["resources/hbr_merge/model1", model2, model3]
)
# merged_model = NormativeModel.load(path="resources/hbr_merge/merged_model")

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.13,95
,2000,0,0.10,63
,2000,0,0.12,127
,2000,0,0.10,319


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.09,31
,2000,0,0.11,31
,2000,0,0.10,319
,2000,0,0.08,63


## Predict with the merged model

We verify that the merged model can handle data from all three groups individually, as well as the full combined dataset.

In [7]:
merged_model.predict(data_group1)
merged_model.predict(data_group2)
merged_model.predict(data_group3)
merged_model.predict(norm_data)

<xarray.NormData> Size: 346kB
Dimensions:            (observations: 1078, response_vars: 2, covariates: 1,
                        batch_effect_dims: 2, centile: 5, statistic: 11)
Coordinates:
  * observations       (observations) int64 9kB 0 1 2 3 ... 1074 1075 1076 1077
  * response_vars      (response_vars) <U23 184B 'WM-hypointensities' 'Right-...
  * covariates         (covariates) <U3 12B 'age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'sex' 'site'
  * centile            (centile) float64 40B 0.05 0.25 0.5 0.75 0.95
  * statistic          (statistic) <U8 352B 'EXPV' 'MACE' ... 'SMSE' 'ShapiroW'
Data variables:
    subject_ids        (observations) object 9kB 'AnnArbor_a_sub04111' ... 'S...
    Y                  (observations, response_vars) float64 17kB 1.687e+03 ....
    X                  (observations, covariates) float64 9kB 25.63 ... 23.0
    batch_effects      (observations, batch_effect_dims) <U17 147kB 'M' ... '...
    Z                  (observations, response_vars) float64 17kB 0.6 ... -0....
    centiles           (centile, observations, response_vars) float64 86kB 77...
    baseline_logp      (observations, response_vars) float64 17kB -1.132 ... ...
    logp               (observations, response_vars) float64 17kB -0.541 ... ...
    Yhat               (observations, response_vars) float64 17kB 1.442e+03 ....
    statistics         (response_vars, statistic) float64 176B 0.08027 ... 0....
Attributes:
    real_ids:                       True
    is_scaled:                      False
    name:                           fcon1000
    unique_batch_effects:           {np.str_('sex'): ['M', 'F'], np.str_('sit...
    batch_effect_counts:            defaultdict(<function NormData.register_b...
    covariate_ranges:               {np.str_('age'): {'min': 7.88, 'max': 85.0}}
    batch_effect_covariate_ranges:  {np.str_('sex'): {'M': {np.str_('age'): {...

## Evaluate

We can inspect the evaluation metrics and visualize the merged model's centiles to confirm it has learned across all contributing sites.

In [8]:
from pcntoolkit import plot_centiles_advanced

# Show evaluation metrics on the full dataset
norm_data.get_statistics_df()

statistic,EXPV,MACE,MAPE,MSLL,NLL,R2,RMSE,Rho,Rho_p,SMSE,ShapiroW
response_vars,,,,,,,,,,,
Right-Lateral-Ventricle,-0.009796,0.066011,0.459191,-0.116740,1.302557,-0.029467,3919.201278,0.301905,3.730233e-24,1.029467,0.874183
WM-hypointensities,0.080269,0.033581,0.336476,-0.454968,1.034626,0.073385,750.086001,0.382271,7.758408e-39,0.926615,0.908901


In [ ]:
# Visualize centile curves from the merged model
plot_centiles_advanced(
    merged_model,
    scatter_data=norm_data,
    batch_effects="all",
)